## Exercise 3: OLTP vs. OLAP Use Cases: Practice with Postgres and DuckDB

In this exercise, we will work to better understand and demonstrate the performance differences between a row-oriented database (PostgreSQL) optimized for OLTP workloads and a column-oriented database (DuckDB) optimized for OLAP workloads. Note that we've already seen some of this in the past, but now we will dive into more detail, and test this system for the OLAP use case as well!

## Visualizing Data Storage
Examine the sample sales data table in this notebook below, which we will use to illustrate the fundamental difference between how row-based and column-based databases store data on disk.

| OrderID | Product  | Quantity | Price | SaleDate   |
|---------|----------|----------|-------|------------|
| 101     | Laptop   | 1        | 1200  | 2025-10-14 |
| 102     | Mouse    | 2        | 25    | 2025-10-14 |
| 103     | Keyboard | 1        | 75    | 2025-10-15 |

 
A) Row-Oriented Storage (PostgreSQL) 

In systems such as PostgreSQL, data for each record is stored together, making it fast to retrieve an entire row. This also makes it substantially faster to write an entire row, given that all the relevant required information is present and provided by the application using the database.

- [101, 'Laptop', 1, 1200, '2025-10-14'] 

- [102, 'Mouse', 2, 25, '2025-10-14'] 

- [103, 'Keyboard', 1, 75, '2025-10-15']

B) Column-Oriented Storage (DuckDB) 

Data is stored by column, making it highly efficient to read and aggregate data from a few specific columns.

- [101, 102, 103] 

- ['Laptop', 'Mouse', 'Keyboard']

- [1, 2, 1] 

- [1200, 25, 75]

- ['2025-10-14', '2025-10-14', '2025-10-15']

We often will see row oriented systems called "OLTP" databases and column oriented systems, called "OLAP" database or data warehouses. 

Question: If you wanted to calculate the total revenue (sum of Quantity * Price), which storage format would likely be faster and why? 

The answer would typically be the column oriented approach, which allows you to ignore the irrelevant fields. Although if you wanted to write data, the row oriented approach would be faster, as you wouldn't have to write to each separate column component.

let's use our Python notebook to connect to PostgreSQL and DuckDB, run benchmark tests, and see the performance difference firsthand. The industry actually has data which can be used for this benchmarking purpose specifically. These datasets are called TPC-C and TPC-H.

- TPC-C: an On-Line Transaction Processing Benchmark - https://www.tpc.org/tpcc/
- TPC-H: a Decision Support Benchmark (OLAP) - https://www.tpc.org/tpch/

The OLTP TPC-C tests are actually quite complex, and involve the creation of multiple simulated users, multiple writes per user, and then the deletion of those users. This can often be done using software such as HammerDB (an industry open source benchmarking tool used to test systems like cloud hosted databases, such as AWS RDS). For now, we've provided scripts that simulate the TPC-C results. You can view their output by running the cells below, and examine the benchmark. 

Let's start once again by importing everything we will need.

**Note: Running this cell will drop tables and allow you to restart fresh**

In [1]:
import duckdb
import pandas as pd
import sys
import os
from IPython.core.interactiveshell import InteractiveShell

# Add parent directory to Python path to access utils module
sys.path.insert(0, os.path.abspath('..'))

# Import our utility functions
from utils.db_utils import create_pg_connection, connect_to_duckdb, ensure_tpcc_schema, execute_query
from utils.tpch_utils import transfer_all_tables
from utils.tpcc_utils import (
    create_tpcc_schema, 
    create_tpcc_indexes,
    generate_mock_data, 
    load_data_postgres, 
    load_data_duckdb,
    run_benchmark,
    run_concurrent_benchmark,
    PG_SQL, 
    DUCK_SQL
)

# Configure display settings to prevent output truncation
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False
%config InteractiveShell.ast_node_interactivity = "all"

# Stop truncating rows (show all)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

# Clean up databases to start fresh
print("Cleaning up databases...")

# 1. Drop PostgreSQL schemas (tpcc and tables from tpch)
try:
    pg_conn = create_pg_connection()
    with pg_conn.cursor() as cursor:
        cursor.execute("DROP SCHEMA IF EXISTS tpcc CASCADE;")
        # Drop TPC-H tables in public schema
        cursor.execute("""DROP SCHEMA IF EXISTS tpch CASCADE;""")
    pg_conn.commit()
    pg_conn.close()
    print("✓ PostgreSQL schemas and tables dropped")
except Exception as e:
    print(f"PostgreSQL cleanup failed: {e}")

# 2. Clean up DuckDB files
import pathlib
for db_file in ['tpcc.duckdb', 'tpch.duckdb']:
    db_path = pathlib.Path(db_file)
    if db_path.exists():
        db_path.unlink()
        print(f"✓ {db_file} removed")

print("Database cleanup complete - starting fresh!\n")

Cleaning up databases...
✓ PostgreSQL schemas and tables dropped
✓ tpcc.duckdb removed
✓ tpch.duckdb removed
Database cleanup complete - starting fresh!



Next, let's load up our sample data! Note this will take a minute or longer!

In [2]:
# --- Configuration ---
SCALE_FACTOR = 2  # Number of warehouses

def main():
    """Main data loading function."""
    print("--- Starting TPC-C Data Load ---")
    
    # 1. Generate Data
    dataframes = generate_mock_data(SCALE_FACTOR)
    
    # 2. Load into PostgreSQL
    pg_conn = None
    try:
        pg_conn = create_pg_connection()
        ensure_tpcc_schema(pg_conn)
        create_tpcc_schema(pg_conn, is_postgres=True)
        load_data_postgres(pg_conn, dataframes)
        # Add indexes to improve OLTP performance
        create_tpcc_indexes(pg_conn)
    except Exception as e:
        print(f"PostgreSQL loading failed: {e}")
    finally:
        if pg_conn:
            pg_conn.close()
            print("PostgreSQL connection closed.")

    # 3. Load into DuckDB
    duck_conn = None
    try:
        duck_conn = connect_to_duckdb(database='tpcc.duckdb')
        create_tpcc_schema(duck_conn, is_postgres=False)
        load_data_duckdb(duck_conn, dataframes)
    except Exception as e:
        print(f"DuckDB loading failed: {e}")
    finally:
        if duck_conn:
            duck_conn.close()
            print("DuckDB connection closed.")
            
    print("\n--- TPC-C Data Load Complete ---")
    print(f"Data scale: {SCALE_FACTOR} warehouse(s)")
    print(f"DuckDB file: tpcc.duckdb")
    print(f"PostgreSQL schema: tpcc")

if __name__ == "__main__":
    main()

--- Starting TPC-C Data Load ---
Generating mock data for 2 warehouse(s)...
Mock data generated.
Successfully ensured 'tpcc' schema exists.
Creating 9 TPC-C tables (PostgreSQL)...
Tables created.
Loading data into PostgreSQL...
  Loading warehouse (2 rows)...
  Loading district (20 rows)...
  Loading customer (60000 rows)...
  Skipping empty table: history
  Loading item (100000 rows)...
  Loading stock (200000 rows)...
  Skipping empty table: orders
  Skipping empty table: new_order
  Skipping empty table: order_line
PostgreSQL load complete in 4.29 seconds.
Creating indexes on PostgreSQL tables...
Indexes created.
PostgreSQL connection closed.
Successfully connected to DuckDB at 'tpcc.duckdb'.
Creating 9 TPC-C tables (DuckDB)...
Tables created.
Loading data into DuckDB...
  Loading warehouse (2 rows)...
  Loading district (20 rows)...
  Loading customer (60000 rows)...
  Skipping empty table: history
  Loading item (100000 rows)...
  Loading stock (200000 rows)...
  Skipping empty ta

Now, let's run our TPC-C simulation benchmark! Note, this is a simplification of what the actual TPC-C setup is, but it's a useful tool for our understanding. This will simulate 8 different streams writing over 400 transactions to the database for purchases, updates, and so on in our e-commerce backend! 

In [3]:
# --- Configuration ---
TOTAL_TRANSACTIONS = 400
NUM_WORKERS = 8  # Number of concurrent connections
# ---------------------

def main():
    """Main concurrent benchmark execution."""
    print("--- Starting TPC-C Transaction Test ---")
    print(f"PostgreSQL will use {NUM_WORKERS} concurrent connections to simulate multiple users. DuckDB will run single-threaded (it doesn't support concurrent writes from multiple processes).")
    
    # Connection factory function for PostgreSQL workers
    def create_pg_conn():
        conn = create_pg_connection()
        with conn.cursor() as cursor:
            cursor.execute("SET search_path TO tpcc, public;")
        return conn
    
    # 1. Run PostgreSQL with concurrency
    try:
        run_concurrent_benchmark(
            "PostgreSQL", 
            create_pg_conn, 
            PG_SQL, 
            is_postgres=True, 
            scale_factor=SCALE_FACTOR, 
            total_transactions=TOTAL_TRANSACTIONS,
            num_workers=NUM_WORKERS
        )
    except Exception as e:
        print(f"PostgreSQL concurrent benchmark failed: {e}")
    
    # 2. Run DuckDB single-threaded
    duck_conn = None
    try:
        duck_conn = connect_to_duckdb(database='tpcc.duckdb')
        run_benchmark("DuckDB (single-threaded)", duck_conn, DUCK_SQL, is_postgres=False, 
                     scale_factor=SCALE_FACTOR, total_transactions=TOTAL_TRANSACTIONS)
    except Exception as e:
        print(f"DuckDB benchmark failed: {e}")
    finally:
        if duck_conn:
            duck_conn.close()
            print("\nDuckDB connection closed.")
    
    print("\n--- TPC-C Test Complete ---")
    print("\nKey Takeaway: PostgreSQL handles concurrent transactions efficiently,")
    print("while DuckDB is limited to single-writer scenarios - this is PostgreSQL's OLTP advantage!")

if __name__ == "__main__":
    main()

--- Starting TPC-C Transaction Test ---
PostgreSQL will use 8 concurrent connections to simulate multiple users. DuckDB will run single-threaded (it doesn't support concurrent writes from multiple processes).

--- Running Concurrent TPC-C Benchmark on PostgreSQL ---
  Workers: 8
  Total Transactions: 400
--- PostgreSQL Concurrent Results ---
  Total Transactions: 400
  Total Time: 6.01 seconds
  Transactions Per Second (TPS): 66.53
  Transaction Mix (Success/Total):
    New Order:    157 / 157
    Payment:      203 / 203
    Order Status: 40 / 40
Successfully connected to DuckDB at 'tpcc.duckdb'.

--- Running TPC-C Benchmark on DuckDB (single-threaded) ---
--- DuckDB (single-threaded) Results ---
  Total Transactions: 400
  Total Time: 13.40 seconds
  Transactions Per Second (TPS): 29.86
  Transaction Mix (Success/Total):
    New Order:    181 / 181
    Payment:      178 / 178
    Order Status: 41 / 41

DuckDB connection closed.

--- TPC-C Test Complete ---

Key Takeaway: PostgreSQL ha

Let's see what that loaded!

In [4]:
# Check what was loaded into the TPC-C database
pg_conn = create_pg_connection()
cursor = pg_conn.cursor()

# Set the search path to use the tpcc schema
cursor.execute("SET search_path TO tpcc, public;")

print("=== TPC-C Data Summary ===\n")

# Count records in each table
tables = ["warehouse", "district", "customer", "item", "stock", "orders", "new_order", "order_line", "history"]
for table in tables:
    cursor.execute(f"SELECT COUNT(*) FROM {table};")
    count = cursor.fetchone()[0]
    print(f"{table.upper()}: {count:,} records")

print("\n=== Sample Warehouse Data ===")
cursor.execute("SELECT * FROM warehouse LIMIT 3;")
for row in cursor.fetchall():
    print(row)

print("\n=== Sample District Data ===")
cursor.execute("SELECT d_id, d_w_id, d_name, d_next_o_id FROM district LIMIT 5;")
for row in cursor.fetchall():
    print(row)

print("\n=== Sample Customer Data ===")
cursor.execute("SELECT c_id, c_d_id, c_w_id, c_first, c_last, c_balance FROM customer LIMIT 5;")
for row in cursor.fetchall():
    print(row)

print("\n=== Sample Item Data ===")
cursor.execute("SELECT i_id, i_name, i_price FROM item LIMIT 5;")
for row in cursor.fetchall():
    print(row)

cursor.close()
pg_conn.close()

=== TPC-C Data Summary ===

WAREHOUSE: 2 records
DISTRICT: 20 records
CUSTOMER: 60,000 records
ITEM: 100,000 records
STOCK: 200,000 records
ORDERS: 157 records
NEW_ORDER: 157 records
ORDER_LINE: 1,518 records
HISTORY: 203 records

=== Sample Warehouse Data ===
(2, 'Smith, Mclaughlin and Ramirez', '15969 Andrew Stream', 'Suite 273', 'New Gary', 'OH', '79657    ', Decimal('0.1506'), Decimal('546878.01'))
(1, 'Church PLC', '3958 May Rue', 'Apt. 322', 'New Jose', 'MS', '66480    ', Decimal('0.1868'), Decimal('553239.61'))

=== Sample District Data ===
(9, 2, 'Robinsonport', 3009)
(3, 2, 'East Kristiefurt', 3011)
(6, 2, 'Blaketon', 3004)
(8, 1, 'Stephanieshire', 3006)
(10, 1, 'Davidfurt', 3008)

=== Sample Customer Data ===
(1, 1, 1, 'Suzanne', 'Malone', Decimal('-10.00'))
(2, 1, 1, 'Jack', 'Bishop', Decimal('-10.00'))
(3, 1, 1, 'Timothy', 'Rosario', Decimal('-10.00'))
(4, 1, 1, 'Billy', 'Johnson', Decimal('-10.00'))
(5, 1, 1, 'Jennifer', 'Trujillo', Decimal('-10.00'))

=== Sample Item Data

Now let's check DuckDB!


In [5]:
# Check what was loaded into DuckDB TPC-C database
duck_conn = connect_to_duckdb(database='tpcc.duckdb', read_only=True)

print("=== DuckDB TPC-C Data Summary ===\n")

# Count records in each table
tables = ["warehouse", "district", "customer", "item", "stock", "orders", "new_order", "order_line", "history"]
for table in tables:
    result = duck_conn.execute(f"SELECT COUNT(*) FROM {table};").fetchone()
    count = result[0]
    print(f"{table.upper()}: {count:,} records")

print("\n=== Sample Warehouse Data ===")
result = duck_conn.execute("SELECT * FROM warehouse LIMIT 3;").fetchall()
for row in result:
    print(row)

print("\n=== Sample District Data ===")
result = duck_conn.execute("SELECT d_id, d_w_id, d_name, d_next_o_id FROM district LIMIT 5;").fetchall()
for row in result:
    print(row)

print("\n=== Sample Customer Data ===")
result = duck_conn.execute("SELECT c_id, c_d_id, c_w_id, c_first, c_last, c_balance FROM customer LIMIT 5;").fetchall()
for row in result:
    print(row)

print("\n=== Sample Item Data ===")
result = duck_conn.execute("SELECT i_id, i_name, i_price FROM item LIMIT 5;").fetchall()
for row in result:
    print(row)

duck_conn.close()

Successfully connected to DuckDB at 'tpcc.duckdb'.
=== DuckDB TPC-C Data Summary ===

WAREHOUSE: 2 records
DISTRICT: 20 records
CUSTOMER: 60,000 records
ITEM: 100,000 records
STOCK: 200,000 records
ORDERS: 181 records
NEW_ORDER: 181 records
ORDER_LINE: 1,822 records
HISTORY: 178 records

=== Sample Warehouse Data ===
(1, 'Church PLC', '3958 May Rue', 'Apt. 322', 'New Jose', 'MS', '66480', 0.1868, 536899.0200000001)
(2, 'Smith, Mclaughlin and Ramirez', '15969 Andrew Stream', 'Suite 273', 'New Gary', 'OH', '79657', 0.1506, 516165.0099999999)

=== Sample District Data ===
(1, 1, 'Morganfurt', 3007)
(2, 1, 'North Sandyville', 3009)
(3, 1, 'North Shannonville', 3014)
(4, 1, 'South Tyler', 3013)
(5, 1, 'Lake Tonya', 3009)

=== Sample Customer Data ===
(1, 1, 1, 'Suzanne', 'Malone', -10.0)
(2, 1, 1, 'Jack', 'Bishop', -10.0)
(3, 1, 1, 'Timothy', 'Rosario', -10.0)
(4, 1, 1, 'Billy', 'Johnson', -10.0)
(5, 1, 1, 'Jennifer', 'Trujillo', -10.0)

=== Sample Item Data ===
(1, 'Soldier speak grow feel

So, we can see the benchmarks are super different between DuckDB and Postgres for transactional use cases. 

Since we've already seen that OLTP use cases are much faster on PostgreSQL from our previous assignment, let us compare OLAP use cases instead.

We can use the industry standard TPC-H setup for that, which already comes with data prepared. You see the TPC-H schema here: https://docs.snowflake.com/en/_images/sample-data-tpch-schema.png

It ultimately simulates data from a "store" which receives many orders from multiple nations and customers, for various products or parts.  

Let's start with DuckDB, the embedded OLAP database, which should come out on top in this comparison. 

In [6]:
%load_ext sql
conn = duckdb.connect(database='tpch.duckdb')
%sql conn --alias duckdb

In [7]:
%%sql
INSTALL tpch;
LOAD tpch;

,Success


The above code will install TPC-H which actually comes built in with functions for generating data on DuckDB.

The TPC-H implementation on DuckDB will generate data for a standard scale of 1 (there are multiple scales used in TPC-H)

In [8]:
%%sql
CALL dbgen(sf = 0.001);

,Success


Now that we have the data for TPC-H, let's see how fast DuckDB is at running the benchmarks.

TPC-H doesn't just come with data generating functions. It also comes with a set of 21 queries which are standard for benchmarking. We can see the queries by running the commands for the extension below:

### Task 1: Select 'query; from tpch_queries table. What does it contain?

In [9]:
%%sql
SELECT 
    REGEXP_REPLACE(
        --- BEGIN SOLUTION
        query, 
        --- END SOLUTION
        '[\r\n]+', ' ', 'g'
    ) AS query
FROM tpch_queries();

,query
0,"SELECT l_returnflag, l_linestatus, sum(l_quantity) AS sum_qty, sum(l_extendedprice) AS sum_base_price, sum(l_extendedprice * (1 - l_discount)) AS sum_disc_price, sum(l_extendedprice * (1 - l_discount) * (1 + l_tax)) AS sum_charge, avg(l_quantity) AS avg_qty, avg(l_extendedprice) AS avg_price, avg(l_discount) AS avg_disc, count(*) AS count_order FROM lineitem WHERE l_shipdate <= CAST('1998-09-02' AS date) GROUP BY l_returnflag, l_linestatus ORDER BY l_returnflag, l_linestatus;"
1,"SELECT s_acctbal, s_name, n_name, p_partkey, p_mfgr, s_address, s_phone, s_comment FROM part, supplier, partsupp, nation, region WHERE p_partkey = ps_partkey AND s_suppkey = ps_suppkey AND p_size = 15 AND p_type LIKE '%BRASS' AND s_nationkey = n_nationkey AND n_regionkey = r_regionkey AND r_name = 'EUROPE' AND ps_supplycost = ( SELECT min(ps_supplycost) FROM partsupp, supplier, nation, region WHERE p_partkey = ps_partkey AND s_suppkey = ps_suppkey AND s_nationkey = n_nationkey AND n_regionkey = r_regionkey AND r_name = 'EUROPE') ORDER BY s_acctbal DESC, n_name, s_name, p_partkey LIMIT 100;"
2,"SELECT l_orderkey, sum(l_extendedprice * (1 - l_discount)) AS revenue, o_orderdate, o_shippriority FROM customer, orders, lineitem WHERE c_mktsegment = 'BUILDING' AND c_custkey = o_custkey AND l_orderkey = o_orderkey AND o_orderdate < CAST('1995-03-15' AS date) AND l_shipdate > CAST('1995-03-15' AS date) GROUP BY l_orderkey, o_orderdate, o_shippriority ORDER BY revenue DESC, o_orderdate LIMIT 10;"
3,"SELECT o_orderpriority, count(*) AS order_count FROM orders WHERE o_orderdate >= CAST('1993-07-01' AS date) AND o_orderdate < CAST('1993-10-01' AS date) AND EXISTS ( SELECT * FROM lineitem WHERE l_orderkey = o_orderkey AND l_commitdate < l_receiptdate) GROUP BY o_orderpriority ORDER BY o_orderpriority;"
4,"SELECT n_name, sum(l_extendedprice * (1 - l_discount)) AS revenue FROM customer, orders, lineitem, supplier, nation, region WHERE c_custkey = o_custkey AND l_orderkey = o_orderkey AND l_suppkey = s_suppkey AND c_nationkey = s_nationkey AND s_nationkey = n_nationkey AND n_regionkey = r_regionkey AND r_name = 'ASIA' AND o_orderdate >= CAST('1994-01-01' AS date) AND o_orderdate < CAST('1995-01-01' AS date) GROUP BY n_name ORDER BY revenue DESC;"
5,SELECT sum(l_extendedprice * l_discount) AS revenue FROM lineitem WHERE l_shipdate >= CAST('1994-01-01' AS date) AND l_shipdate < CAST('1995-01-01' AS date) AND l_discount BETWEEN 0.05 AND 0.07 AND l_quantity < 24;
6,"SELECT supp_nation, cust_nation, l_year, sum(volume) AS revenue FROM ( SELECT n1.n_name AS supp_nation, n2.n_name AS cust_nation, extract(year FROM l_shipdate) AS l_year, l_extendedprice * (1 - l_discount) AS volume FROM supplier, lineitem, orders, customer, nation n1, nation n2 WHERE s_suppkey = l_suppkey AND o_orderkey = l_orderkey AND c_custkey = o_custkey AND s_nationkey = n1.n_nationkey AND c_nationkey = n2.n_nationkey AND ((n1.n_name = 'FRANCE' AND n2.n_name = 'GERMANY') OR (n1.n_name = 'GERMANY' AND n2.n_name = 'FRANCE')) AND l_shipdate BETWEEN CAST('1995-01-01' AS date) AND CAST('1996-12-31' AS date)) AS shipping GROUP BY supp_nation, cust_nation, l_year ORDER BY supp_nation, cust_nation, l_year;"
7,"SELECT o_year, sum( CASE WHEN nation = 'BRAZIL' THEN volume ELSE 0 END) / sum(volume) AS mkt_share FROM ( SELECT extract(year FROM o_orderdate) AS o_year, l_extendedprice * (1 - l_discount) AS volume, n2.n_name AS nation FROM part, supplier, lineitem, orders, customer, nation n1, nation n2, region WHERE p_partkey = l_partkey AND s_suppkey = l_suppkey AND l_orderkey = o_orderkey AND o_custkey = c_custkey AND c_nationkey = n1.n_nationkey AND n1.n_regionkey = r_regionkey AND r_name = 'AMERICA' AND s_nationkey = n2.n_nationkey AND o_orderdate BETWEEN CAST('1995-01-01' AS date) AND CAST('1996-12-31' AS date) AND p_type = 'ECONOMY ANODIZED STEEL') AS all_nations GROUP BY o_year ORDER BY o_year;"
8,"SELECT nation, o_year, sum(amount) AS sum_profit FROM ( SELECT n_name AS nation, extract(year FROM o_orde

You can see the queries are largely selects, with several joins. Let's give them a run!

In [10]:
%%timeit
%%sql
PRAGMA tpch(1);
PRAGMA tpch(2);
PRAGMA tpch(3);
PRAGMA tpch(4);
PRAGMA tpch(5);
PRAGMA tpch(6);
PRAGMA tpch(7);
PRAGMA tpch(8);
PRAGMA tpch(9);
PRAGMA tpch(10);
PRAGMA tpch(11);
PRAGMA tpch(12);
PRAGMA tpch(13);
PRAGMA tpch(14);
PRAGMA tpch(15);
PRAGMA tpch(16);
PRAGMA tpch(17);
PRAGMA tpch(18);
PRAGMA tpch(19);
PRAGMA tpch(20);
PRAGMA tpch(21);

85.5 ms ± 8.21 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


We now have an estimate for how long typical analytics / aggregation / read queries take on an OLAP embedded database. Now let's see how Postgres handles this.

Note - this isn't an apples to apples comparison, or a fair benchmark. It's largely just to illustrate the strengths of OLAP vs. OLTP use cases. There are many variables we haven't controlled here, such as network delay (DuckDB is embedded, Postgres is not), machine/server memory (DuckDB runs locally, Postgres runs on a server) and so on. But, you'll still see a difference! Let's start by creating the tables in PostgreSQL.

Next, let's insert data that we have into PostgreSQL.

In [11]:
# Main execution
# Close any existing DuckDB connection first
try:
    if 'conn' in globals():
        conn.close()
except:
    pass

pg_conn = create_pg_connection()
duck_conn = connect_to_duckdb(database='tpch.duckdb', read_only=False)

transfer_all_tables(duck_conn, pg_conn)

Successfully connected to DuckDB at 'tpch.duckdb'.
Starting TPC-H data transfer from DuckDB to PostgreSQL (schema 'tpch')...
Successfully ensured 'tpch' schema exists.

--- Processing table: region ---
  Extracting 'region' from DuckDB...
  Extracted 5 rows in 0.00 seconds.
  Generating schema for 'region'...
  Loading 'region' into PostgreSQL schema 'tpch' using COPY...
  Successfully loaded 'region' into 'tpch' in 0.03 seconds.

--- Processing table: nation ---
  Extracting 'nation' from DuckDB...
  Extracted 25 rows in 0.00 seconds.
  Generating schema for 'nation'...
  Loading 'nation' into PostgreSQL schema 'tpch' using COPY...
  Successfully loaded 'nation' into 'tpch' in 0.03 seconds.

--- Processing table: part ---
  Extracting 'part' from DuckDB...
  Extracted 200 rows in 0.00 seconds.
  Generating schema for 'part'...
  Loading 'part' into PostgreSQL schema 'tpch' using COPY...
  Successfully loaded 'part' into 'tpch' in 0.01 seconds.

--- Processing table: supplier ---
  Ext

0.22550559043884277

Finally, let's run the queries in PostgreSQL! Go ahead and load up the query string we will need. 

#### Task 2: Import TPCH_QUERY_STRING from the utils library we've prepared.


In [12]:
## You can start this statement with from utils.tpch_utils import ....
### BEGIN SOLUTION HERE
from utils.tpch_utils import TPCH_QUERY_STRING
### END SOLUTION HERE

In [13]:
%%timeit -n 3
%%capture 
execute_query(conn=create_pg_connection(), sql_query=TPCH_QUERY_STRING, fetch=True)

593 ms ± 170 ms per loop (mean ± std. dev. of 7 runs, 3 loops each)


You can see that DuckDB is much faster than Postgres on the same type of query for analytical reads.

#### Task 3: Why is DuckDB so much faster than Postgres for analytical reads that involve large aggregations over many elements, and Postgres is better at fetching/writing individual rows?

This is freeform content